# SQL Complete Practice — Part 07: SQL Joins
**Topics 121–131 | MySQL | Ankanhaldar**

This notebook explains how to retrieve related data from two or more tables using SQL joins. Each join type has its own theory and SQL example.

> **Execution note:** These are MySQL SQL cells, not Python. Run them in MySQL Workbench, VS Code with SQLTools, or a compatible SQL notebook/kernel. A standard Python kernel does not execute SQL directly.

> **Practice database:** Run the setup cell first. The sample database contains departments, employees, and projects.

In [ ]:
CREATE DATABASE IF NOT EXISTS ankan_sql_practice;
USE ankan_sql_practice;

CREATE TABLE IF NOT EXISTS join_demo_department (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(80) NOT NULL
);

CREATE TABLE IF NOT EXISTS join_demo_employee (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(80) NOT NULL,
    department_id INT NULL,
    manager_id INT NULL,
    salary DECIMAL(10, 2),
    CONSTRAINT fk_join_demo_employee_department
        FOREIGN KEY (department_id)
        REFERENCES join_demo_department(department_id)
);

CREATE TABLE IF NOT EXISTS join_demo_project (
    project_id INT PRIMARY KEY,
    project_name VARCHAR(100) NOT NULL,
    department_id INT
);

INSERT INTO join_demo_department (department_id, department_name)
VALUES
    (10, 'IT'),
    (20, 'HR'),
    (30, 'Finance'),
    (40, 'Marketing')
ON DUPLICATE KEY UPDATE department_name = VALUES(department_name);

INSERT INTO join_demo_employee
    (employee_id, employee_name, department_id, manager_id, salary)
VALUES
    (1, 'Amit', 10, NULL, 60000),
    (2, 'Riya', 20, 1, 45000),
    (3, 'Suman', 10, 1, 55000),
    (4, 'Priya', 30, 1, 52000),
    (5, 'Rahul', NULL, 2, 35000),
    (6, 'Neha', 20, 2, 48000)
ON DUPLICATE KEY UPDATE
    employee_name = VALUES(employee_name),
    department_id = VALUES(department_id),
    manager_id = VALUES(manager_id),
    salary = VALUES(salary);

INSERT INTO join_demo_project (project_id, project_name, department_id)
VALUES
    (101, 'Website Upgrade', 10),
    (102, 'Recruitment Portal', 20),
    (103, 'Budget Automation', 30),
    (104, 'Brand Campaign', 40)
ON DUPLICATE KEY UPDATE
    project_name = VALUES(project_name),
    department_id = VALUES(department_id);

SELECT * FROM join_demo_department;
SELECT * FROM join_demo_employee;
SELECT * FROM join_demo_project;

**Setup note:** The inserts refresh the fixed sample records on reruns. The sample data intentionally includes an employee without a department and a department without employees, so you can observe unmatched rows in outer joins.

## Topic 121: What Is a SQL JOIN?

A **JOIN** combines rows from two or more tables using a related column or join condition.

Why joins are useful:
- Relational databases commonly store related information in separate tables.
- Joins retrieve related information together.
- They reduce unnecessary duplication in table design.

General syntax:

```sql
SELECT columns
FROM table_a
JOIN table_b
    ON table_a.common_column = table_b.common_column;
```

Always use a correct join condition. A missing or incorrect condition can produce a Cartesian product or incorrect matches.

### SQL Example — Topic 121

In [ ]:
SELECT
    e.employee_id,
    e.employee_name,
    d.department_name
FROM join_demo_employee AS e
INNER JOIN join_demo_department AS d
    ON e.department_id = d.department_id;

**Expected result / explanation**

This query combines employee records with their matching department names. Employees with no matching department are not included because this is an inner join.

## Topic 122: INNER JOIN

`INNER JOIN` returns only rows where the join condition matches in both tables.

If a row in either table has no matching row in the other table, that row is excluded from the result.

It is the most commonly used join when only related records are needed.

### SQL Example — Topic 122

In [ ]:
SELECT
    e.employee_id,
    e.employee_name,
    d.department_name
FROM join_demo_employee AS e
INNER JOIN join_demo_department AS d
    ON e.department_id = d.department_id
ORDER BY e.employee_id;

**Expected result / explanation**

Only employees with a valid matching department are returned. Rahul is excluded because his `department_id` is `NULL`.

## Topic 123: LEFT JOIN / LEFT OUTER JOIN

`LEFT JOIN` returns every row from the left table and matching rows from the right table.

- Matching rows show values from both tables.
- If no right-side match exists, right-side columns return `NULL`.

`LEFT OUTER JOIN` is another way to write `LEFT JOIN`.

### SQL Example — Topic 123

In [ ]:
SELECT
    d.department_id,
    d.department_name,
    e.employee_name
FROM join_demo_department AS d
LEFT JOIN join_demo_employee AS e
    ON d.department_id = e.department_id
ORDER BY d.department_id, e.employee_id;

**Expected result / explanation**

Every department is returned. Marketing appears even though it has no employees; its employee column is `NULL`.

## Topic 124: RIGHT JOIN / RIGHT OUTER JOIN

`RIGHT JOIN` returns every row from the right table and matching rows from the left table.

- Matching rows show values from both tables.
- If no left-side match exists, left-side columns return `NULL`.

`RIGHT OUTER JOIN` is another way to write `RIGHT JOIN`. Many queries can be rewritten as a `LEFT JOIN` by swapping the table order.

### SQL Example — Topic 124

In [ ]:
SELECT
    e.employee_name,
    d.department_id,
    d.department_name
FROM join_demo_employee AS e
RIGHT JOIN join_demo_department AS d
    ON e.department_id = d.department_id
ORDER BY d.department_id, e.employee_id;

**Expected result / explanation**

Every department is returned, including Marketing even though it has no matching employee.

## Topic 125: CROSS JOIN

`CROSS JOIN` returns the Cartesian product of two tables: every row from the first table is paired with every row from the second table.

If the first table has `m` rows and the second has `n` rows, the result contains `m × n` rows.

A cross join does not need a join condition. Use it carefully because the result can become very large.

### SQL Example — Topic 125

In [ ]:
SELECT
    e.employee_name,
    d.department_name
FROM join_demo_employee AS e
CROSS JOIN join_demo_department AS d
ORDER BY e.employee_id, d.department_id
LIMIT 12;

**Expected result / explanation**

The query shows the first 12 combinations of employees and departments. The full Cartesian product would contain 6 × 4 = 24 rows.

## Topic 126: SELF JOIN

A **self join** joins a table to itself. It is useful when rows in the same table have relationships, such as employees and their managers.

A self join uses aliases to distinguish the two references to the same table. The join condition connects an employee's manager ID to another employee's ID.

### SQL Example — Topic 126

In [ ]:
SELECT
    employee.employee_name AS employee_name,
    manager.employee_name AS manager_name
FROM join_demo_employee AS employee
LEFT JOIN join_demo_employee AS manager
    ON employee.manager_id = manager.employee_id
ORDER BY employee.employee_id;

**Expected result / explanation**

The table is used twice: once for employees and once for managers. Employees without a manager are still shown because the query uses `LEFT JOIN`.

## Topic 127: FULL OUTER JOIN and MySQL Workaround

A **FULL OUTER JOIN** returns:
- All matching rows from both tables.
- Unmatched rows from the left table.
- Unmatched rows from the right table.

MySQL does not directly support the `FULL OUTER JOIN` syntax. A common workaround is to combine a `LEFT JOIN` and a `RIGHT JOIN` using `UNION` (or use a left join plus unmatched right rows).

`UNION` removes duplicate result rows. `UNION ALL` keeps duplicates, so the unmatched-right portion must be filtered carefully.

### SQL Example — Topic 127

In [ ]:
-- MySQL FULL OUTER JOIN workaround
SELECT
    d.department_id,
    d.department_name,
    e.employee_name
FROM join_demo_department AS d
LEFT JOIN join_demo_employee AS e
    ON d.department_id = e.department_id

UNION

SELECT
    d.department_id,
    d.department_name,
    e.employee_name
FROM join_demo_department AS d
RIGHT JOIN join_demo_employee AS e
    ON d.department_id = e.department_id;

**Expected result / explanation**

The combined result includes departments without employees and employees without a department. `UNION` removes duplicate rows that were already matched in the first query.

## Topic 128: NATURAL JOIN

`NATURAL JOIN` automatically joins tables using all columns that have the same names and compatible types in both tables.

It can make a query shorter, but it is often discouraged in production code because:
- A new same-named column can silently change the join condition.
- The join condition is not explicit.
- It may be unclear to readers which columns are being matched.

An explicit `JOIN ... ON` is usually easier to maintain.

### SQL Example — Topic 128

In [ ]:
SELECT
    employee_id,
    employee_name,
    department_name
FROM join_demo_employee
NATURAL JOIN join_demo_department;

**Expected result / explanation**

This example uses the common column name `department_id`. The query joins on that same-named column automatically. Use this form only when the shared-column design is intentional and understood.

## Topic 129: JOIN Multiple Tables

A query can join more than two tables by adding another join clause for each related table.

For every join:
- Choose the tables needed for the result.
- Identify the relationship columns.
- Use aliases to keep the query readable.
- Select only the columns required.

The order of joins and the join types affect which rows are retained.

### SQL Example — Topic 129

In [ ]:
SELECT
    e.employee_id,
    e.employee_name,
    d.department_name,
    p.project_name
FROM join_demo_employee AS e
INNER JOIN join_demo_department AS d
    ON e.department_id = d.department_id
INNER JOIN join_demo_project AS p
    ON d.department_id = p.department_id
ORDER BY e.employee_id;

**Expected result / explanation**

The query links employees to departments and departments to projects. Only employees whose department has a matching project are returned.

## Topic 130: JOIN with WHERE and ORDER BY

A join can be combined with `WHERE` to filter matched rows and `ORDER BY` to sort the final result.

Logical query processing applies the join source and conditions before filtering and sorting. A filter on the right-side table in the `WHERE` clause can remove unmatched rows from a `LEFT JOIN`, making it behave like an inner join for that condition.

If you want to preserve unmatched left rows while restricting right-side matches, consider putting the condition in the `ON` clause.

### SQL Example — Topic 130

In [ ]:
SELECT
    e.employee_name,
    d.department_name,
    e.salary
FROM join_demo_employee AS e
LEFT JOIN join_demo_department AS d
    ON e.department_id = d.department_id
WHERE e.salary >= 45000
ORDER BY e.salary DESC;

-- Preserve all departments while matching only employees earning at least 45000
SELECT
    d.department_name,
    e.employee_name,
    e.salary
FROM join_demo_department AS d
LEFT JOIN join_demo_employee AS e
    ON d.department_id = e.department_id
   AND e.salary >= 45000
ORDER BY d.department_id;

**Expected result / explanation**

The first query filters employees by salary and sorts them highest first. The second keeps every department, but only attaches employee rows that satisfy the salary condition.

## Topic 131: JOIN vs SUBQUERY

A **JOIN** combines columns from related tables in one result. A **subquery** is a query nested inside another SQL statement.

Use a join when you need columns from related tables. Use a subquery when the problem is naturally expressed as a membership test, existence check, or comparison with an aggregate.

Many problems can be solved with either approach. The clearest and most efficient form depends on the data, indexes, database optimizer, and query requirements. Do not assume one is always faster.

### SQL Example — Topic 131

In [ ]:
-- JOIN approach: employees whose salary is above the overall average
SELECT
    e.employee_name,
    e.salary
FROM join_demo_employee AS e
JOIN (
    SELECT AVG(salary) AS average_salary
    FROM join_demo_employee
) AS a
    ON e.salary > a.average_salary;

-- Subquery approach: same business question
SELECT
    employee_name,
    salary
FROM join_demo_employee
WHERE salary > (
    SELECT AVG(salary)
    FROM join_demo_employee
);

**Expected result / explanation**

Both queries identify employees whose salary is above the average salary across the employee table. Compare their readability and execution plans on larger datasets.

## JOIN Summary Table

| Join type | Rows returned |
|---|---|
| `INNER JOIN` | Only matching rows from both tables |
| `LEFT JOIN` | All left rows and matching right rows |
| `RIGHT JOIN` | All right rows and matching left rows |
| `CROSS JOIN` | Every possible row combination |
| `SELF JOIN` | A table joined to itself |
| `FULL OUTER JOIN` | All matched and unmatched rows from both sides; use a workaround in MySQL |
| `NATURAL JOIN` | Automatically joins on all same-named compatible columns |

## Visual Reminder

For two tables A and B:
- INNER: intersection (matching rows).
- LEFT: all A rows, plus matches from B.
- RIGHT: all B rows, plus matches from A.
- FULL: all rows from A and B, matched where possible.
- CROSS: every A row paired with every B row.

## Important Interview Points

1. `ON` defines how rows are matched.
2. `WHERE` filters rows after the joined row source is formed.
3. A condition on the right table in `WHERE` can remove NULL-extended rows from a `LEFT JOIN`.
4. MySQL does not provide direct `FULL OUTER JOIN` syntax.
5. A self join uses aliases to refer to the same table more than once.
6. A cross join produces `m × n` combinations.
7. Explicit `JOIN ... ON` is generally clearer than `NATURAL JOIN`.

## Practice Checklist

- [ ] Retrieve employee names with department names using INNER JOIN.
- [ ] Show all departments, including those without employees.
- [ ] Show all employees, including those without departments.
- [ ] Generate combinations using CROSS JOIN.
- [ ] Display each employee with their manager using SELF JOIN.
- [ ] Write a FULL OUTER JOIN workaround in MySQL.
- [ ] Join three related tables.
- [ ] Combine JOIN with WHERE and ORDER BY.
- [ ] Solve one problem using both JOIN and a subquery.

**End of Part 07 — Topics 121–131.**